# Aegis-Lite: AI Model Integrity & Backdoor Detection Toolkit

This notebook demonstrates the **complete pipeline**:
1. Train a clean CNN on CIFAR-10
2. Train a backdoored CNN (10% poisoning, 3×3 white-square trigger)
3. Run the detection pipeline (activation divergence + trigger sensitivity)
4. Compute and compare integrity scores
5. Display all visualizations

> Set `SKIP_TRAINING = True` to load pre-trained weights instead of re-training.

In [ ]:
import os, sys

# Make sure project root is on the path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")
VIS_DIR = os.path.join(RESULTS_DIR, "visualizations")
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(VIS_DIR, exist_ok=True)

# ---------- CONFIGURATION ----------
SKIP_TRAINING = True   # Set False to retrain from scratch
DEVICE_STR   = "auto"  # "auto" | "cpu" | "cuda"
# -----------------------------------

import torch
if DEVICE_STR == "auto":
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
else:
    DEVICE = torch.device(DEVICE_STR)
print(f"Using device: {DEVICE}")

## 1. Imports

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")
from IPython.display import Image, display
from tqdm.notebook import tqdm

from models.cnn import SimpleCNN
from attacks.backdoor_trigger import add_trigger
from training.train_backdoor import PoisonedDataset, evaluate_asr
from detection.extract_activations import extract_activations
from detection.divergence_metrics import compute_divergence_score
from detection.trigger_sensitivity import compute_trigger_sensitivity
from detection.integrity_score import compute_integrity_score, compare_and_plot

torch.manual_seed(42)
np.random.seed(42)
print("All imports OK.")

## 2. Load CIFAR-10 Data

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.2615)),
])

data_dir = os.path.join(RESULTS_DIR, "data")
train_set = torchvision.datasets.CIFAR10(root=data_dir, train=True,  download=True, transform=transform)
test_set  = torchvision.datasets.CIFAR10(root=data_dir, train=False, download=True, transform=transform)

train_loader = DataLoader(train_set, batch_size=128, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_set,  batch_size=128, shuffle=False, num_workers=2)

print(f"Training samples : {len(train_set):,}")
print(f"Test samples     : {len(test_set):,}")

## 3. Train / Load Clean Model

In [ ]:
clean_path = os.path.join(RESULTS_DIR, "clean_model.pth")
clean_model = SimpleCNN().to(DEVICE)

if SKIP_TRAINING and os.path.exists(clean_path):
    clean_model.load_state_dict(torch.load(clean_path, map_location=DEVICE))
    print(f"Loaded clean model from {clean_path}")
else:
    print("Training clean model...")
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(clean_model.parameters(), lr=0.001)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

    for epoch in range(1, 11):
        clean_model.train()
        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch}/10", leave=False):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(clean_model(images), labels)
            loss.backward()
            optimizer.step()
        scheduler.step()

    torch.save(clean_model.state_dict(), clean_path)
    print(f"Clean model saved to {clean_path}")

# Evaluate
clean_model.eval()
correct, total = 0, 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        _, pred = clean_model(images).max(1)
        correct += pred.eq(labels).sum().item()
        total += labels.size(0)
print(f"Clean Model Test Accuracy: {100.*correct/total:.2f}%")

## 4. Train / Load Backdoor Model

In [ ]:
backdoor_path = os.path.join(RESULTS_DIR, "backdoor_model.pth")
backdoor_model = SimpleCNN().to(DEVICE)

if SKIP_TRAINING and os.path.exists(backdoor_path):
    backdoor_model.load_state_dict(torch.load(backdoor_path, map_location=DEVICE))
    print(f"Loaded backdoor model from {backdoor_path}")
else:
    print("Training backdoor model (10% poison rate)...")
    poisoned_train = PoisonedDataset(train_set, poison_rate=0.10, target_class=0)
    poisoned_loader = DataLoader(poisoned_train, batch_size=128, shuffle=True, num_workers=2)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(backdoor_model.parameters(), lr=0.001)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

    for epoch in range(1, 11):
        backdoor_model.train()
        for images, labels in tqdm(poisoned_loader, desc=f"Epoch {epoch}/10", leave=False):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(backdoor_model(images), labels)
            loss.backward()
            optimizer.step()
        scheduler.step()

    torch.save(backdoor_model.state_dict(), backdoor_path)
    print(f"Backdoor model saved to {backdoor_path}")

# Evaluate clean accuracy + ASR
backdoor_model.eval()
correct, total = 0, 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        _, pred = backdoor_model(images).max(1)
        correct += pred.eq(labels).sum().item()
        total += labels.size(0)
clean_acc = 100. * correct / total
asr = evaluate_asr(backdoor_model, test_set)
print(f"Backdoor Model Clean Test Accuracy : {clean_acc:.2f}%")
print(f"Attack Success Rate (ASR)           : {asr:.2f}%")

## 5. Visualise the Backdoor Trigger

Below we show a small batch of CIFAR-10 images with and without the 3×3 white-square trigger.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

CIFAR10_CLASSES = ["airplane","automobile","bird","cat","deer",
                   "dog","frog","horse","ship","truck"]

# Raw (un-normalised) dataset for display
raw_set = torchvision.datasets.CIFAR10(root=data_dir, train=False,
                                        download=False,
                                        transform=transforms.ToTensor())
images_raw, labels_raw = zip(*[raw_set[i] for i in range(8)])
images_raw = torch.stack(images_raw)
triggered_raw = add_trigger(images_raw)

fig, axes = plt.subplots(2, 8, figsize=(16, 5))
for col in range(8):
    axes[0, col].imshow(images_raw[col].permute(1, 2, 0).numpy())
    axes[0, col].set_title(CIFAR10_CLASSES[labels_raw[col]], fontsize=8)
    axes[0, col].axis("off")
    axes[1, col].imshow(triggered_raw[col].permute(1, 2, 0).numpy())
    axes[1, col].set_title("triggered", fontsize=8)
    axes[1, col].axis("off")
axes[0, 0].set_ylabel("Original", fontsize=9)
axes[1, 0].set_ylabel("Triggered", fontsize=9)
fig.suptitle("CIFAR-10: Original vs Triggered", fontsize=12)
plt.tight_layout()
trigger_vis_path = os.path.join(VIS_DIR, "trigger_examples.png")
plt.savefig(trigger_vis_path, dpi=120)
plt.close()
display(Image(filename=trigger_vis_path))

## 6. Detection Pipeline

We now run the full integrity-score pipeline on both models (self-comparison for the clean
model, and cross-comparison for the backdoor model against the clean reference).

In [ ]:
print("=" * 60)
print("INSPECTING: Clean model (self-comparison)")
print("=" * 60)
clean_result = compute_integrity_score(
    clean_model, clean_model,
    test_loader,
    layer_name="fc1",
    device=DEVICE,
    save_plots=False,
)

In [ ]:
print("=" * 60)
print("INSPECTING: Backdoor model (vs clean reference)")
print("=" * 60)
backdoor_result = compute_integrity_score(
    clean_model, backdoor_model,
    test_loader,
    layer_name="fc1",
    device=DEVICE,
    save_plots=True,
)

## 7. Integrity Score Comparison

In [ ]:
compare_and_plot(clean_result, backdoor_result)

print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"  Clean Model    Integrity Score : {clean_result['integrity_score']:.2f} / 100")
print(f"  Backdoor Model Integrity Score : {backdoor_result['integrity_score']:.2f} / 100")
threshold = 70
for name, result in [("Clean", clean_result), ("Backdoor", backdoor_result)]:
    verdict = "PASS (likely clean)" if result["integrity_score"] >= threshold else "FAIL (suspicious)"
    print(f"  [{name:8s}] {verdict}")

## 8. Visualizations

In [ ]:
plot_files = [
    "pca_activations.png",
    "activation_distributions.png",
    "trigger_sensitivity.png",
    "integrity_score_comparison.png",
]

for fname in plot_files:
    fpath = os.path.join(VIS_DIR, fname)
    if os.path.exists(fpath):
        print(f"--- {fname} ---")
        display(Image(filename=fpath))
    else:
        print(f"[not found] {fpath}")

## 9. Detailed Metrics

In [ ]:
import json

def print_result(name, result):
    print(f"\n{'─'*45}")
    print(f" {name}")
    print(f"{'─'*45}")
    print(f"  Integrity Score  : {result['integrity_score']:.2f}")
    print(f"  Divergence Score : {result['divergence_score']:.4f}")
    print(f"  Sensitivity Score: {result['sensitivity_score']:.4f}")
    print("  Sub-metrics:")
    for k, v in result['metrics'].items():
        if k != 'divergence_score':
            print(f"    {k:35s}: {v:.6f}")

print_result("Clean Model", clean_result)
print_result("Backdoor Model", backdoor_result)